# PRCP-1012 · Game Winner Prediction (PUBG)

**Problem statement**

- **Task 1** – Prepare a complete data analysis report on the given data.
- **Task 2** – Create a predictive model that predicts the **win probability** (`winPlacePerc`) of a PUBG match and find the important factors affecting it.
- **Model Comparison Report** – compare multiple models and suggest the best one for production.
- **Report on Challenges** – challenges faced with the data and the techniques used, with reasons.

**Target:** `winPlacePerc` – percentile winning placement, 1 = 1st place (winner), 0 = last place.

| Section | Contents |
|---|---|
| 1 | Setup & data loading |
| 2 | Task 1 – Data analysis report (EDA) |
| 3 | Data cleaning & anomaly (cheater) detection |
| 4 | Feature engineering |
| 5 | Task 2 – Model building & evaluation |
| 6 | Feature importance – factors affecting win probability |
| 7 | Model comparison report |
| 8 | Report on challenges faced |
| 9 | Conclusion & saving the model |

## 1. Setup & data loading

In [ ]:
import warnings, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import joblib

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 60)
RANDOM_STATE = 42
TARGET = "winPlacePerc"

**Dataset:** download `PRCP-1012-GameWinnerPred.zip` from the link in the brief, extract it and put `train_V2.csv` in the project's `data/` folder.
The full file has ~4.45 M rows, so we sample matches (`SAMPLE_MATCHES`) to keep training time reasonable. Set it to `None` to use everything.

If the real file is missing, the notebook falls back to `data/sample_pubg.csv` (synthetic data with the same schema, created by `src/generate_sample_data.py`) **only so the code can be tested** – results on that file are not valid project results.

In [ ]:
DATA_DIR = Path("../data") if Path("../data").exists() else Path("data")
CANDIDATES = ["train_V2.csv", "pubg.csv", "PUBG_Game_Prediction_data.csv"]
SAMPLE_MATCHES = 8000      # None = use all ~47,000 matches

def reduce_memory(df):
    """Downcast numeric dtypes: float64/int64 -> smallest type that fits."""
    before = df.memory_usage(deep=True).sum() / 1024**2
    for c in df.select_dtypes("number").columns:
        kind = "integer" if pd.api.types.is_integer_dtype(df[c]) else "float"
        df[c] = pd.to_numeric(df[c], downcast=kind)
    after = df.memory_usage(deep=True).sum() / 1024**2
    print(f"Memory: {before:,.1f} MB -> {after:,.1f} MB ({100*(1-after/before):.0f}% saved)")
    return df

path = next((DATA_DIR / f for f in CANDIDATES if (DATA_DIR / f).exists()), None)
SYNTHETIC = path is None
if SYNTHETIC:
    path = DATA_DIR / "sample_pubg.csv"
    print("WARNING: real dataset not found - using SYNTHETIC sample for a test run.")
raw = reduce_memory(pd.read_csv(path))
print("Loaded", path, raw.shape)

In [ ]:
if SAMPLE_MATCHES and raw["matchId"].nunique() > SAMPLE_MATCHES:
    keep = pd.Series(raw["matchId"].unique()).sample(SAMPLE_MATCHES, random_state=RANDOM_STATE)
    df = raw[raw["matchId"].isin(keep)].reset_index(drop=True)
else:
    df = raw.copy()
print(f"Working data: {df.shape[0]:,} rows | {df['matchId'].nunique():,} matches | {df['groupId'].nunique():,} groups")
df.head()

## 2. Task 1 – Data analysis report

### 2.1 Structure, data types and summary statistics

In [ ]:
df.info()

In [ ]:
df.describe().T.style.format("{:,.2f}")

**Attribute dictionary**

| Column | Meaning |
|---|---|
| Id, groupId, matchId | Player, team and match identifiers |
| assists | Enemies damaged by the player that a team-mate killed |
| boosts / heals | Boost items / healing items used |
| damageDealt | Total damage dealt (self-damage excluded) |
| DBNOs | Enemies knocked down |
| headshotKills, kills, roadKills, teamKills | Kill counts by type |
| killPlace | Rank in the match by kills |
| killStreaks | Max enemies killed in a short time |
| longestKill | Longest kill distance (m) |
| matchDuration | Match length (s) |
| matchType | Game mode (solo/duo/squad, fpp/tpp, events) |
| maxPlace / numGroups | Worst placement recorded / number of groups with data |
| killPoints, rankPoints, winPoints | External Elo-like ratings (inconsistent – see 2.2) |
| revives | Team-mates revived |
| rideDistance, swimDistance, walkDistance | Distance travelled (m) |
| vehicleDestroys | Vehicles destroyed |
| weaponsAcquired | Weapons picked up |
| **winPlacePerc** | **Target** – placement percentile (1 = win) |

*Note:* the brief lists `maxPlace` twice; the second one is `numGroups` in the actual file.

### 2.2 Missing values, duplicates and inconsistent columns

In [ ]:
miss = df.isna().sum()
print("Missing values:\n", miss[miss > 0] if miss.any() else "none")
print("\nDuplicate Ids:", df["Id"].duplicated().sum())
print("Rows with rankPoints == -1 :", (df["rankPoints"] == -1).mean().round(3), "share")
print("Rows with killPoints == 0  :", (df["killPoints"] == 0).mean().round(3), "share")
print("Rows with winPoints == 0   :", (df["winPoints"] == 0).mean().round(3), "share")

**Observation:** the target has one missing row (a match with a single player), which is dropped.
`rankPoints` is `-1` (= "None") for a large share of players and `killPoints`/`winPoints` are `0` exactly when `rankPoints` is filled – the three rating systems are mutually exclusive and mostly empty, so they are dropped.

### 2.3 Target variable – `winPlacePerc`

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.histplot(df[TARGET].dropna(), bins=50, ax=ax[0], color="#4C72B0")
ax[0].set_title("Distribution of winPlacePerc")
sns.boxplot(x=df[TARGET].dropna(), ax=ax[1], color="#55A868")
ax[1].set_title("Boxplot of winPlacePerc")
plt.tight_layout(); plt.show()
print(df[TARGET].describe().round(3))

The target is roughly **uniform on [0, 1]** – by construction every placement from first to last exists in each match. Peaks at 0 and 1 are the last-place and winning groups.

### 2.4 Match types

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
mt = df["matchType"].value_counts()
sns.barplot(x=mt.values, y=mt.index, ax=ax[0], color="#4C72B0")
ax[0].set_title("Players per matchType")
MODE = {"solo":"solo","solo-fpp":"solo","normal-solo":"solo","normal-solo-fpp":"solo",
        "duo":"duo","duo-fpp":"duo","normal-duo":"duo","normal-duo-fpp":"duo",
        "squad":"squad","squad-fpp":"squad","normal-squad":"squad","normal-squad-fpp":"squad"}
df["matchMode"] = df["matchType"].map(MODE).fillna("event")
df["matchMode"].value_counts().plot.pie(autopct="%1.1f%%", ax=ax[1], ylabel="")
ax[1].set_title("Grouped game mode")
plt.tight_layout(); plt.show()

**Observation:** squad-fpp and duo-fpp dominate. The 16 raw types were grouped into 4 modes (solo / duo / squad / event) because rare types (crash, flare, normal-*) have too few rows to learn from separately.

### 2.5 Univariate analysis of key features

In [ ]:
feats = ["kills", "damageDealt", "walkDistance", "rideDistance", "boosts", "heals",
         "weaponsAcquired", "killPlace", "longestKill", "DBNOs", "assists", "revives"]
fig, axes = plt.subplots(3, 4, figsize=(18, 11))
for ax, f in zip(axes.ravel(), feats):
    clip = df[f].quantile(0.99)
    sns.histplot(df[f].clip(upper=clip), bins=40, ax=ax, color="#4C72B0")
    ax.set_title(f"{f} (clipped at p99={clip:.0f})", fontsize=10)
plt.tight_layout(); plt.show()

**Observations**
- Almost all count features are **heavily right-skewed** – most players get 0 kills, use few items and die early.
- `walkDistance` has a long tail: players who survive longer walk much further.
- `rideDistance` is 0 for most players – vehicles are used by a minority.
- `killPlace` is uniform by design (it is a rank).

In [ ]:
print("Players with 0 kills  : {:.1%}".format((df.kills == 0).mean()))
print("Players with 0 damage : {:.1%}".format((df.damageDealt == 0).mean()))
print("Players never driving : {:.1%}".format((df.rideDistance == 0).mean()))
print("Players walking 0 m   : {:.1%}".format((df.walkDistance == 0).mean()))

### 2.6 Bivariate analysis – features vs. win probability

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
pairs = [("kills", 12), ("boosts", 12), ("heals", 15), ("weaponsAcquired", 15), ("DBNOs", 10), ("headshotKills", 8)]
for ax, (f, cap) in zip(axes.ravel(), pairs):
    tmp = df[[f, TARGET]].dropna().copy()
    tmp[f] = tmp[f].clip(upper=cap)
    sns.boxplot(x=f, y=TARGET, data=tmp, ax=ax, color="#8172B3", fliersize=0)
    ax.set_title(f"winPlacePerc vs {f} (capped at {cap}+)")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
for ax, f in zip(axes, ["walkDistance", "damageDealt", "killPlace"]):
    s = df[[f, TARGET]].dropna().sample(min(30000, len(df)), random_state=1)
    ax.scatter(s[f].clip(upper=s[f].quantile(.99)), s[TARGET], s=3, alpha=.25, color="#C44E52")
    ax.set_xlabel(f); ax.set_ylabel(TARGET); ax.set_title(f"{f} vs winPlacePerc")
plt.tight_layout(); plt.show()

**Observations**
- **walkDistance** has the clearest relationship with winning – surviving longer means walking further. It is effectively a proxy for survival time.
- More **kills, boosts, heals and weapons acquired** steadily raise the median placement.
- **killPlace** is strongly *negatively* related: a lower kill rank (more kills relative to others) means a better finish.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
sns.boxplot(x="matchMode", y=TARGET, data=df, ax=ax[0], order=["solo","duo","squad","event"])
ax[0].set_title("winPlacePerc by game mode")
df["groupSize"] = df.groupby(["matchId","groupId"])["Id"].transform("count")
sns.boxplot(x=df["groupSize"].clip(upper=6), y=df[TARGET], ax=ax[1], color="#64B5CD")
ax[1].set_title("winPlacePerc by group size (6 = 6+)")
plt.tight_layout(); plt.show()

### 2.7 Correlation analysis

In [ ]:
num = df.select_dtypes("number").drop(columns=["groupSize"], errors="ignore")
corr = num.corr()
plt.figure(figsize=(16, 12))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=True, fmt=".2f", annot_kws={"size": 7}, linewidths=.3)
plt.title("Correlation matrix"); plt.show()

In [ ]:
top = corr[TARGET].drop(TARGET).sort_values()
plt.figure(figsize=(9, 7))
top.plot.barh(color=np.where(top > 0, "#55A868", "#C44E52"))
plt.title("Correlation of each feature with winPlacePerc"); plt.axvline(0, color="k", lw=.8); plt.show()
print(top.sort_values(ascending=False).round(3).head(8))

**Correlation summary**
- Strongest positive: **walkDistance**, **boosts**, **weaponsAcquired**, **damageDealt**, **heals**, **kills**.
- Strongest negative: **killPlace**.
- `kills`, `damageDealt`, `DBNOs`, `headshotKills` and `killStreaks` are inter-correlated (multicollinearity) – tree models handle this well; linear models do not.
- `killPoints`, `rankPoints`, `winPoints`, `matchDuration` and `teamKills` are nearly uncorrelated with the target.

### 2.8 Data analysis summary (Task 1)

1. ~4.45 M player rows, 29 columns; one missing target value; no duplicate Ids.
2. Target is uniform in [0, 1] and is **shared by all members of a group**, so prediction is really at group level.
3. Features are strongly right-skewed with long tails, and contain physically impossible values (handled in section 3).
4. Survival-related behaviour (walking, boosting, healing, collecting weapons) drives placement more than aggression alone.
5. The rating columns (`killPoints`, `rankPoints`, `winPoints`) are inconsistent and mostly empty → dropped.
6. Placement is **relative to the other groups in the same match**, which motivates in-match rank features.

## 3. Data cleaning & anomaly (cheater) detection

In [ ]:
df = df[df[TARGET].notna()].drop_duplicates("Id")
df["totalDistance"] = df["walkDistance"] + df["rideDistance"] + df["swimDistance"]
df["headshotRate"] = (df["headshotKills"] / df["kills"]).replace([np.inf, -np.inf], 0).fillna(0)

rules = {
    "kills without moving":       (df.kills > 0) & (df.totalDistance == 0),
    "roadKills > 10":             df.roadKills > 10,
    "kills > 40":                 df.kills > 40,
    "100% headshots, 10+ kills":  (df.headshotRate == 1) & (df.kills > 9),
    "longestKill >= 1000 m":      df.longestKill >= 1000,
    "walkDistance >= 10 km":      df.walkDistance >= 10000,
    "rideDistance >= 20 km":      df.rideDistance >= 20000,
    "swimDistance >= 2 km":       df.swimDistance >= 2000,
    "weaponsAcquired >= 80":      df.weaponsAcquired >= 80,
    "heals >= 40":                df.heals >= 40,
}
summary = pd.DataFrame({"rows_flagged": {k: int(v.sum()) for k, v in rules.items()}})
display(summary)
anomaly = np.logical_or.reduce(list(rules.values()))
print(f"Total anomalous rows removed: {anomaly.sum():,} ({anomaly.mean():.3%})")
df = df[~anomaly].drop(columns=["killPoints", "rankPoints", "winPoints"]).reset_index(drop=True)
print("Clean shape:", df.shape)

**Why these rules?** They describe records that are impossible for a fair player (e.g. kills without moving a single metre, sniping beyond render distance, 100 % headshots over 10+ kills). Such rows are almost always cheaters or logging bugs and would teach the model wrong relationships. They are a tiny fraction of the data, so removing them costs nothing in coverage.

## 4. Feature engineering

In [ ]:
df["playersJoined"] = df.groupby("matchId")["matchId"].transform("count")
norm = (100 - df["playersJoined"]) / 100 + 1
df["killsNorm"]      = df["kills"] * norm           # kills are easier in full lobbies -> normalise
df["damageNorm"]     = df["damageDealt"] * norm
df["healsAndBoosts"] = df["heals"] + df["boosts"]
df["killStreakRate"] = (df["killStreaks"] / df["kills"]).replace([np.inf, -np.inf], 0).fillna(0)
df["walkPerKill"]    = df["walkDistance"] / (df["kills"] + 1)
df["boostsPerWalk"]  = df["boosts"] / (df["walkDistance"] + 1)
df["healsPerWalk"]   = df["heals"] / (df["walkDistance"] + 1)
df["weaponsPerWalk"] = df["weaponsAcquired"] / (df["walkDistance"] + 1)
df["killPlacePerc"]  = df["killPlace"] / df.groupby("matchId")["killPlace"].transform("max")
df["distancePerSec"] = df["totalDistance"] / df["matchDuration"].clip(lower=1)

new_feats = ["playersJoined","killsNorm","damageNorm","healsAndBoosts","totalDistance","headshotRate",
             "killStreakRate","walkPerKill","boostsPerWalk","healsPerWalk","weaponsPerWalk","killPlacePerc","distancePerSec"]
df[new_feats + [TARGET]].corr()[TARGET].drop(TARGET).sort_values(ascending=False).round(3)

### 4.1 Group-level aggregation and in-match ranking

All players in a group share one `winPlacePerc`. Training on individual rows would (a) repeat the same label up to 4× and (b) ignore that a team wins together. So we aggregate each group's players into **mean / max / min** of every feature, add **group size**, and add the **percentile rank** of the group's mean and max stats **within its match** – placement is decided against the other groups in that same match.

In [ ]:
BASE = ["assists","boosts","damageDealt","DBNOs","headshotKills","heals","killPlace","kills","killStreaks",
        "longestKill","revives","rideDistance","roadKills","swimDistance","teamKills","vehicleDestroys",
        "walkDistance","weaponsAcquired"] + [f for f in new_feats if f != "playersJoined"]
MATCH_LEVEL = ["matchDuration", "maxPlace", "numGroups", "playersJoined"]

g = df.groupby(["matchId", "groupId"])
X = pd.concat([g[BASE].mean().add_suffix("_mean"), g[BASE].max().add_suffix("_max"),
               g[BASE].min().add_suffix("_min"), g.size().rename("groupSize"),
               g[MATCH_LEVEL].first()], axis=1)
rank_cols = [c for c in X.columns if c.endswith(("_mean", "_max"))]
X = X.join(X.groupby(level="matchId")[rank_cols].rank(pct=True).add_suffix("_rank"))
mode = g["matchMode"].first()
for m in ["solo", "duo", "squad", "event"]:
    X[f"mode_{m}"] = (mode == m).astype(np.int8)
X = X.astype(np.float32)
y = g[TARGET].first()
print("Group-level feature matrix:", X.shape)

## 5. Task 2 – Model building & evaluation

**Split strategy:** `GroupShuffleSplit` on `matchId` (80/20). Groups from the same match must not appear in both train and test, otherwise the in-match rank features leak information and scores look unrealistically good.

**Metric:** Mean Absolute Error (MAE) – same unit as the target, robust to outliers, official metric for this problem. RMSE and R² are secondary.

In [ ]:
match_ids = X.index.get_level_values("matchId")
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE).split(X, y, groups=match_ids))
X_tr, X_te, y_tr, y_te = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
print(f"Train groups: {len(X_tr):,} | Test groups: {len(X_te):,} | Features: {X.shape[1]}")

**Post-processing.** Within each match the true targets are evenly spaced placements. After predicting, we rank groups inside each match by predicted score and map ranks back onto the valid placement grid `(maxPlace − place) / (maxPlace − 1)`. This keeps predictions in [0, 1], removes ties and usually lowers MAE.

In [ ]:
def postprocess(pred, groups):
    t = groups[["maxPlace"]].copy(); t["pred"] = pred
    t["rank"] = t.groupby(level="matchId")["pred"].rank(method="first")
    n = t.groupby(level="matchId")["pred"].transform("count")
    adj = (t["rank"] - 1) / (n - 1).replace(0, 1)
    gap = 1 / (t["maxPlace"] - 1).replace(0, 1)
    adj = np.round(adj / gap) * gap
    adj[t["maxPlace"] <= 1] = 0
    return adj.clip(0, 1).values

def evaluate(name, model):
    t0 = time.time(); model.fit(X_tr, y_tr); secs = time.time() - t0
    p = np.clip(model.predict(X_te), 0, 1); pp = postprocess(p, X_te)
    r = {"Model": name, "MAE": mean_absolute_error(y_te, p),
         "RMSE": np.sqrt(mean_squared_error(y_te, p)), "R2": r2_score(y_te, p),
         "MAE_postprocessed": mean_absolute_error(y_te, pp), "Train_seconds": round(secs, 1)}
    r["Best_MAE"] = min(r["MAE"], r["MAE_postprocessed"])
    print(f"{name:18s} MAE={r['MAE']:.4f}  MAE(pp)={r['MAE_postprocessed']:.4f}  R2={r['R2']:.4f}  {secs:.1f}s")
    return r, model

In [ ]:
models = {
    "Linear Regression": make_pipeline(StandardScaler(), LinearRegression()),
    "Ridge Regression":  make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "Decision Tree":     DecisionTreeRegressor(max_depth=12, min_samples_leaf=20, random_state=RANDOM_STATE),
    "Random Forest":     RandomForestRegressor(n_estimators=100, max_depth=18, min_samples_leaf=5, max_features=0.4,
                                               max_samples=0.5, n_jobs=-1, random_state=RANDOM_STATE),
    "XGBoost":           XGBRegressor(n_estimators=600, learning_rate=0.05, max_depth=7, subsample=0.8,
                                      colsample_bytree=0.7, tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE),
    "LightGBM":          LGBMRegressor(n_estimators=800, learning_rate=0.05, num_leaves=63, subsample=0.8, subsample_freq=1,
                                       colsample_bytree=0.7, n_jobs=-1, random_state=RANDOM_STATE, verbose=-1),
}
results, fitted = [], {}
for name, m in models.items():
    r, fitted[name] = evaluate(name, m); results.append(r)
res = pd.DataFrame(results).sort_values("Best_MAE").reset_index(drop=True)
res.style.format({c: "{:.4f}" for c in ["MAE","RMSE","R2","MAE_postprocessed","Best_MAE"]}).highlight_min(subset=["Best_MAE"], color="#c7e9c0")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
o = res.sort_values("Best_MAE", ascending=False)
ax[0].barh(o.Model, o.MAE, color="#8C8C8C", label="raw"); ax[0].barh(o.Model, o.MAE_postprocessed, color="#4C72B0", alpha=.7, label="post-processed")
ax[0].set_title("MAE (lower is better)"); ax[0].legend()
ax[1].barh(o.Model, o.R2, color="#55A868"); ax[1].set_title("R² (higher is better)")
ax[2].barh(o.Model, o.Train_seconds, color="#DD8452"); ax[2].set_title("Training time (s)")
plt.tight_layout(); plt.show()

In [ ]:
best_name = res.loc[0, "Model"]; best = fitted[best_name]
use_pp = bool(res.loc[0, "MAE_postprocessed"] < res.loc[0, "MAE"])
pred = np.clip(best.predict(X_te), 0, 1)
if use_pp: pred = postprocess(pred, X_te)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
ax[0].scatter(y_te, pred, s=3, alpha=.3); ax[0].plot([0,1],[0,1],"r--")
ax[0].set_xlabel("Actual"); ax[0].set_ylabel("Predicted"); ax[0].set_title(f"{best_name}: actual vs predicted")
sns.histplot(pred - y_te, bins=60, ax=ax[1], color="#8172B3"); ax[1].set_title("Residuals")
plt.tight_layout(); plt.show()
print("Best model:", best_name, "| post-processing:", use_pp)

## 6. Feature importance – factors affecting win probability

In [ ]:
gbm = fitted["LightGBM"]
imp = pd.Series(gbm.booster_.feature_importance("gain"), index=X.columns).sort_values(ascending=False)
imp = imp / imp.sum()
plt.figure(figsize=(10, 9))
imp.head(25)[::-1].plot.barh(color="#4C72B0")
plt.title("Top 25 features (LightGBM, gain share)"); plt.show()

In [ ]:
# Roll importance up to the original game statistic (sum over mean/max/min/rank variants)
base_of = lambda c: c.replace("_rank","").rsplit("_", 1)[0] if c.endswith(("_mean","_max","_min","_rank")) else c
by_stat = imp.groupby(imp.index.map(base_of)).sum().sort_values(ascending=False)
plt.figure(figsize=(10, 7))
by_stat.head(15)[::-1].plot.barh(color="#55A868")
plt.title("Importance by underlying game statistic"); plt.show()
by_stat.head(10).round(3)

**Interpretation – what drives the win probability**

1. **Walking distance / total distance** – the strongest factor; it is a direct proxy for how long a player stayed alive.
2. **Kill place & kills (in-match rank)** – out-fighting the other groups in the same match matters more than raw kill counts.
3. **Boosts and heals** – using items lets players survive the shrinking zone and late fights.
4. **Weapons acquired** – looting well is linked to surviving longer and winning fights.
5. **Damage dealt / knock-downs** – combat effectiveness, especially for squads.
6. **Group size & match context** (numGroups, maxPlace, mode) – calibrate what a given placement means.

**Practical takeaway:** play for survival (rotate with the zone, keep boosts/heals topped up, loot efficiently) and win the fights you take – passive survival plus efficient combat beats chasing kills.

## 7. Model comparison report

In [ ]:
display(res.style.format({c: "{:.4f}" for c in ["MAE","RMSE","R2","MAE_postprocessed","Best_MAE"]}))
print(f"Recommended production model: {best_name}  (MAE={res.loc[0,'Best_MAE']:.4f}, R2={res.loc[0,'R2']:.4f})")

| Model | Strengths | Weaknesses |
|---|---|---|
| Linear Regression | Fast, interpretable baseline | Cannot capture non-linear interactions; affected by multicollinearity |
| Ridge Regression | Handles multicollinearity via L2 penalty | Still linear |
| Decision Tree | Non-linear, interpretable rules | Overfits, step-like predictions, highest variance |
| Random Forest | Robust, low variance | Slow to train and very large on 4.4 M rows |
| XGBoost | High accuracy, regularised boosting | Slower than LightGBM, more memory |
| **LightGBM** | **Best accuracy/speed trade-off, histogram binning, low memory, native feature importance** | Needs tuning of `num_leaves` to avoid overfitting |

**Recommendation for production: LightGBM + rank-based post-processing.**

- Lowest (or joint-lowest) MAE among the models on the PUBG data; boosted trees capture the non-linear interactions between survival, combat and item-use features.
- Trains on millions of rows in minutes and predicts in milliseconds; model file is small.
- Needs no feature scaling and handles skewed features natively.
- Built-in feature importance gives explainability for stakeholders.

If the table above shows a different winner on a small sample (e.g. on the synthetic test data), the margin is usually within a few thousandths of MAE; LightGBM is still preferred for production because it scales best to the full dataset. The same table is saved to `reports/model_comparison.csv` by `src/train.py`.

## 8. Report on challenges faced

| # | Challenge | Technique used | Reason |
|---|---|---|---|
| 1 | **Large dataset** (~4.45 M rows, ~1 GB in RAM) | Downcast numeric dtypes; sample matches (not rows) for experiments | Cuts memory by more than half; sampling whole matches keeps every match complete so in-match features stay valid |
| 2 | **Missing target** (1 row) | Dropped the row | A match with one player has no defined placement; a single row has no impact on the data |
| 3 | **Inconsistent rating columns** (`killPoints`, `rankPoints`, `winPoints`) | Dropped them | Mutually exclusive, mostly `-1`/`0`, near-zero correlation with the target – keeping them adds noise |
| 4 | **Cheaters / impossible records** | Rule-based anomaly filters (kills without moving, ≥1 km kills, 100 % headshots on 10+ kills, extreme distances, item counts) | These rows break the real relationships between behaviour and placement; they are a tiny fraction of the data |
| 5 | **Highly skewed features & outliers** | Tree-based models; ratio features (per-walk, per-kill); clipping only for plots | Trees are invariant to monotonic transforms and robust to outliers, so heavy transformation is unnecessary |
| 6 | **16 sparse match types** | Grouped into solo / duo / squad / event and one-hot encoded | Rare types (crash, flare, normal-*) have too few rows; game mode is what changes team dynamics |
| 7 | **Target shared by group members** | Aggregated players to group level (mean/max/min + group size) | Avoids duplicate labels and models the team as the unit that actually places |
| 8 | **Placement is relative to the match** | Percentile rank of group stats inside each match; lobby-size normalisation of kills/damage | A strong stat in a weak lobby means more than in a strong one; ranks capture relative strength |
| 9 | **Data leakage risk** | `GroupShuffleSplit` by `matchId` | Groups from one match share rank features; splitting by match gives an honest test score |
| 10 | **Multicollinearity** (kills, damage, DBNOs, headshots) | Ridge for the linear baseline; tree ensembles for final models | L2 stabilises linear coefficients; trees are unaffected |
| 11 | **Predictions outside [0, 1] / off the placement grid** | Clipping + rank-based post-processing per match | Enforces valid, evenly spaced placements and lowers MAE |
| 12 | **Training time** | LightGBM histogram boosting; `max_samples` for Random Forest | Keeps experiments fast enough to compare six models |

## 9. Conclusion & saving the model

- **Task 1:** the analysis shows a uniform target shared by group members, skewed behavioural features, a small number of cheater records and unusable rating columns. Survival-related behaviour (walking, boosting, healing, looting) is most associated with winning.
- **Task 2:** after cleaning, feature engineering and group-level aggregation, six models were compared. **LightGBM with rank-based post-processing** is recommended for production; walk distance, in-match kill rank, boosts/heals, weapons acquired and damage dealt are the key factors affecting win probability.

In [ ]:
out = Path("../models") if Path("../models").exists() else Path("models")
out.mkdir(exist_ok=True)
joblib.dump({"model": best, "features": list(X.columns), "name": best_name, "use_postprocess": use_pp},
            out / "best_model_notebook.joblib")
print("Saved", out / "best_model_notebook.joblib")
if SYNTHETIC:
    print("\nReminder: these results come from the SYNTHETIC sample. Put train_V2.csv in data/ and re-run all cells.")